# EDA — Base de Escola (Bronze `br_inep_censo_escolar.escola`)

Depois da decisão de pivô de granularidade (aluno → escola, documentada em `reports/decisoes.md`), preciso entender de verdade os dados que vou usar como base do modelo daqui pra frente. Este notebook tem um objetivo bem específico e mais restrito do que o `01_eda.ipynb`: **não é ainda uma EDA completa, é um diagnóstico de colunas**. Trago as 455 colunas da tabela Bronze `escola` sem filtrar nada, e o foco é medir o percentual de preenchimento de cada uma antes de decidir quais entram na próxima etapa (pendência 13). Só faz sentido fazer uma EDA mais profunda depois, olhando só as colunas que sobrarem — é por isso que separei isso num notebook à parte, em vez de misturar com a EDA de aluno.

Aproveito e já monto as três colunas candidatas a variável-alvo que eu tinha em mente: quantidade de alunos avaliados por escola, percentual de alunos aprovados, e o peso de cada escola na base total (em número de alunos). No final, faço uma correlação simples (Spearman) entre cada coluna numérica da `escola` e esse percentual de aprovados, só pra ter uma primeira leitura de quais colunas parecem mais relevantes.

**Observações importantes antes de rodar:**
- Isso roda em paralelo ao `01_eda.ipynb` (Opção A, nível aluno), que continua sendo o entregável do Dia 2 — este notebook aqui é o começo do Dia 3 (Opção C, nível escola).
- Vou combinar os alunos de 2023 e 2024 juntos no cálculo do alvo por escola (a tabela `escola` só existe pra `ano=2024`, e eu já tinha decidido aplicar essa safra pros dois anos de aluno). Isso é uma simplificação para este diagnóstico inicial — a forma definitiva de tratar a dimensão `ano` no alvo continua sendo a pendência 11, que ainda vou fechar mais pra frente.
- Uso `alfabetizado` (a mesma variável-alvo que uso desde o Dia 1) como a definição de "aprovado" pra calcular o percentual por escola — isso inclui os ausentes/não avaliados como "não aprovados", seguindo a definição atual de `alfabetizado`. Documento aqui essa escolha como um pressuposto assumido conscientemente: "aprovados" poderia significar outra coisa (ex. aprovação de série, e não alfabetização), mas optei por manter a mesma variável-alvo que já vinha usando desde o início, por consistência com o resto do projeto — se essa leitura precisar mudar mais pra frente, volto e reviso este trecho.
- Isso vai consumir bastante memória (455 colunas x ~215 mil escolas) — fechar outros programas pesados antes de rodar ajuda.

In [ ]:
import sys
sys.path.append("../..")

import boto3
import pandas as pd
import numpy as np

from src.preprocessing.load_data import BUCKET, _ler_parquet_do_prefixo, ler_alunos

pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 50)

print(f"Bucket: {BUCKET}")


## 1. Path confirmado da tabela `escola` no S3

Path confirmado no `catalogo-dados-fase3.md` atualizado (seção 7, que passou a listar os paths físicos do S3 pra Bronze e Silver — na primeira versão do catálogo eu só tinha o nome da tabela no BigQuery, sem o path real, por isso o notebook tinha células de "descoberta" aqui antes):

```
s3://pos-tech-fiap-985034838182-us-east-1-an/bronze/br_inep_censo_escolar/escola/
```

O catálogo também confirma duas coisas que eu já assumia por outra fonte: só existe `ano=2024` (não é particionado por ano, é uma safra só), e `id_escola` é a 4ª coluna da tabela — a chave que vou usar pra fazer o JOIN com `alunos` na Seção 4.

In [ ]:
# path confirmado no catalogo-dados-fase3.md, seção 7
PREFIXO_BRONZE_ESCOLA = "bronze/br_inep_censo_escolar/escola/"

escola = _ler_parquet_do_prefixo(BUCKET, PREFIXO_BRONZE_ESCOLA)
print(f"escola (Bronze, todas as colunas): {escola.shape[0]:,} linhas, {escola.shape[1]} colunas")
print(f"Uso de memória: {escola.memory_usage(deep=True).sum() / 1e6:,.1f} MB")

### ⚠️ Achado crítico (rodado em 2026-09-07): só vieram 11 colunas, não 455

O catálogo (seção 4) descreve a tabela `escola` com 455 colunas (água, energia, esgoto, acessibilidade, equipamentos, corpo técnico, matrícula por sexo/raça/idade/zona/turno, docentes, etc.). Mas a leitura real do path confirmado trouxe só **11 colunas**: `ano, sigla_uf, id_municipio, id_escola, rede` + os mesmos 6 indicadores binários de infraestrutura que já uso desde o Dia 1 (`agua_potavel, internet, equipamento_computador, biblioteca, laboratorio_informatica, esgoto_rede_publica`).

Isso é uma discrepância séria: **a justificativa inteira do pivô pra escola (Opção C) foi a riqueza das 455 colunas** — principalmente a demografia de matrícula por sexo/raça/idade (seção 4.7 do catálogo), que é exatamente o que falta na Opção A. Sem essas colunas, a escola não tem vantagem real sobre continuar no grão de aluno/município. Antes de continuar qualquer análise nessa direção, preciso confirmar: o catálogo descreveu o schema *teórico* disponível no BigQuery de origem, mas só uma parte foi de fato copiada pro S3? Ou existe outro path/arquivo com o restante dos dados que não peguei aqui? Checo abaixo os arquivos brutos do S3 (nome e tamanho) e as pastas vizinhas, pra investigar antes de assumir qualquer coisa.

In [ ]:
# quantos arquivos existem nesse prefixo, e qual o tamanho deles? um arquivo
# com só 11 colunas x 215 mil linhas deveria ser bem pequeno - se o(s)
# arquivo(s) for(em) muito maior(es) que isso, pode ser sinal de que tem
# mais coluna "escondida" que o pandas não juntou certo
s3 = boto3.client("s3")
paginador = s3.get_paginator("list_objects_v2")

arquivos_encontrados = []
for pagina in paginador.paginate(Bucket=BUCKET, Prefix=PREFIXO_BRONZE_ESCOLA):
    for objeto in pagina.get("Contents", []):
        arquivos_encontrados.append((objeto["Key"], objeto["Size"]))

print(f"Arquivos encontrados no prefixo {PREFIXO_BRONZE_ESCOLA}:")
for chave, tamanho in arquivos_encontrados:
    print(f"  {chave} — {tamanho / 1e6:.2f} MB")

print(f"\nColunas realmente lidas: {escola.columns.tolist()}")

# também vale conferir se existe alguma pasta IRMÃ (ex. um nome parecido com
# "escola_completo" ou "escola_raw") dentro do mesmo dataset, que eu possa
# ter deixado passar
resposta_dataset = s3.list_objects_v2(
    Bucket=BUCKET, Prefix="bronze/br_inep_censo_escolar/", Delimiter="/"
)
print("\nPastas dentro de bronze/br_inep_censo_escolar/:")
for prefixo in resposta_dataset.get("CommonPrefixes", []):
    print(" -", prefixo["Prefix"])

## 2. Diagnóstico de preenchimento por coluna

Antes de qualquer correlação ou tratamento, quero ver, coluna por coluna: tipo de dado, quantidade/percentual de nulo, quantidade de valores únicos. É esse resultado que vai me dizer quais das 455 colunas realmente valem a pena manter (pendência 13) — colunas com preenchimento muito baixo provavelmente saem, mas prefiro decidir isso olhando o número real, não supondo de antemão.

In [ ]:
resumo_colunas = pd.DataFrame({
    "coluna": escola.columns,
    "dtype": escola.dtypes.astype(str).values,
    "qtd_preenchido": escola.notnull().sum().values,
    "qtd_nulo": escola.isnull().sum().values,
    "pct_nulo": (escola.isnull().mean() * 100).round(2).values,
    "valores_unicos": [escola[c].nunique(dropna=True) for c in escola.columns],
})
resumo_colunas = resumo_colunas.sort_values("pct_nulo").reset_index(drop=True)

print(f"Total de colunas: {len(resumo_colunas)}")
print(f"Colunas 100% preenchidas (0% nulo): {(resumo_colunas['pct_nulo'] == 0).sum()}")
print(f"Colunas com pelo menos 50% de nulo: {(resumo_colunas['pct_nulo'] >= 50).sum()}")
print(f"Colunas completamente vazias (100% nulo): {(resumo_colunas['pct_nulo'] == 100).sum()}")

resumo_colunas


### 2.1 Export em texto (CSV) pra colar na conversa

Uso `to_csv` direto pro `print`, sem salvar arquivo — só pra eu poder copiar a saída daqui e colar em outro lugar (ex.: numa conversa) sem precisar anexar arquivo.

In [ ]:
print(resumo_colunas.to_csv(index=False))


## 3. Colunas candidatas a variável-alvo (agregadas a partir de `alunos`)

Combino os alunos avaliados de 2023 e 2024 (ver observação no topo do notebook) e agrego por `id_escola`:
- `quantidade_alunos_escola`: quantos alunos avaliados essa escola teve, no total dos dois anos
- `percentual_aprovados`: % desses alunos com `alfabetizado = 1` (nota: isso conta os ausentes/não avaliados como "não aprovados", seguindo a definição atual de `alfabetizado` — a pendência 1, sobre tratar `presenca`/`preenchimento_caderno` de outro jeito, pode mudar esse número depois)
- `peso_escola_pct`: quanto essa escola representa do total de alunos avaliados na base inteira (`quantidade_alunos_escola` dividido pela soma de todas as escolas, x 100)

In [ ]:
alunos = ler_alunos()

# alfabetizado às vezes vem como string/código ("0"/"1") - garanto que é
# numérico antes de agregar, senão a média sai errada silenciosamente (ou
# quebra) em vez de dar um erro claro
alunos["alfabetizado"] = alunos["alfabetizado"].astype(int)

# id_escola precisa estar no mesmo tipo dos dois lados do merge que vou
# fazer na Seção 4 - já converto aqui pra string pra não correr o mesmo
# risco de merge silenciosamente vazio que já aconteceu antes com `rede`
alunos["id_escola"] = alunos["id_escola"].astype(str)

agregado_escola = (
    alunos.groupby("id_escola")
    .agg(
        quantidade_alunos_escola=("alfabetizado", "size"),
        percentual_aprovados=("alfabetizado", lambda s: s.mean() * 100),
    )
    .reset_index()
)

total_alunos_base = agregado_escola["quantidade_alunos_escola"].sum()
agregado_escola["peso_escola_pct"] = (
    agregado_escola["quantidade_alunos_escola"] / total_alunos_base * 100
)

print(f"Escolas com pelo menos 1 aluno avaliado (2023+2024): {agregado_escola.shape[0]:,}")
print(f"Total de alunos avaliados (2023+2024): {total_alunos_base:,}")
agregado_escola.describe()


## 4. Juntando com a tabela `escola` e checando a interseção (prévia da pendência 12)

Faço um `LEFT JOIN` da `escola` (Bronze) com o agregado acima, mantendo TODAS as escolas do Censo (mesmo as sem nenhum aluno avaliado) — assim dá pra ver de cara qual fração das ~215 mil escolas do Censo realmente tem o alvo calculável, e também o inverso (alunos cuja escola não aparece no Censo).

In [ ]:
escola["id_escola"] = escola["id_escola"].astype(str)

escola_com_alvo = escola.merge(agregado_escola, on="id_escola", how="left")

tem_alvo = escola_com_alvo["percentual_aprovados"].notna()
print(f"Escolas no Censo (Bronze): {escola_com_alvo.shape[0]:,}")
print(f"Escolas com alvo calculável (têm aluno avaliado): {tem_alvo.sum():,} ({tem_alvo.mean() * 100:.1f}%)")
print(f"Escolas SEM nenhum aluno avaliado (alvo fica nulo): {(~tem_alvo).sum():,} ({(~tem_alvo).mean() * 100:.1f}%)")

# checagem inversa: tem aluno com id_escola que não bate com nenhuma escola do Censo?
escolas_no_censo = set(escola["id_escola"])
alunos_sem_escola_no_censo = ~alunos["id_escola"].isin(escolas_no_censo)
print(f"\nAlunos cujo id_escola NÃO aparece no Censo Escolar: {alunos_sem_escola_no_censo.sum():,} "
      f"({alunos_sem_escola_no_censo.mean() * 100:.2f}% dos alunos avaliados)")


### ⚠️ Achado crítico (rodado em 2026-09-07): 0% de interseção em `id_escola`

O JOIN acima deu **0% de correspondência nos dois sentidos** — nenhuma escola do Censo bateu com nenhum aluno avaliado. Isso é forte demais pra ser "a interseção real é baixa" (o esperado seria uma fração menor, não um zero absoluto) — um zero absoluto desse tipo costuma ser diferença de **formato** da chave (zeros à esquerda, dígito verificador a mais/a menos, string vs. número armazenado como texto diferente), não ausência real de interseção. Antes de aceitar que essa tabela não serve pra Opção C, comparo o formato bruto dos valores dos dois lados.

In [ ]:
# comparo os valores brutos de id_escola dos dois lados - as conversões de
# tipo que já apliquei acima (.astype(str)) podem estar mascarando uma
# diferença de formato, então releio direto das tabelas originais
amostra_escola = escola["id_escola"].head(10).tolist()
amostra_alunos = alunos["id_escola"].head(10).tolist()

print("Amostra de id_escola na tabela `escola` (Bronze):")
print(amostra_escola)
print("Tamanhos (nº de caracteres) mais comuns:")
print(escola["id_escola"].astype(str).str.len().value_counts().head())

print("\nAmostra de id_escola na tabela `alunos` (Silver):")
print(amostra_alunos)
print("Tamanhos (nº de caracteres) mais comuns:")
print(alunos["id_escola"].astype(str).str.len().value_counts().head())

### 4.1 Consolidando os dois diagnósticos num bloco só

Junto a saída das duas células de diagnóstico acima (arquivos/pastas no S3 + amostra de `id_escola`) num bloco só, pra ter um registro único e organizado dessa investigação — fica mais fácil documentar no `reports/decisoes.md` e também consultar depois sem precisar rodar tudo de novo. Não recalculo nada aqui — só reaproveito as variáveis que as duas células anteriores já deixaram prontas, então essa célula só funciona se as duas de cima já tiverem rodado antes dela.

In [ ]:
print("=" * 70)
print("DIAGNÓSTICO 1 — arquivos brutos no S3 e colunas realmente lidas")
print("=" * 70)

print(f"\nPrefixo lido: {PREFIXO_BRONZE_ESCOLA}")
print("Arquivos encontrados:")
for chave, tamanho in arquivos_encontrados:
    print(f"  {chave} — {tamanho / 1e6:.2f} MB")

print(f"\nColunas realmente lidas ({len(escola.columns)} de 455 esperadas pelo catálogo):")
print(escola.columns.tolist())

print("\nPastas dentro de bronze/br_inep_censo_escolar/:")
for prefixo in resposta_dataset.get("CommonPrefixes", []):
    print(" -", prefixo["Prefix"])

print("\n" + "=" * 70)
print("DIAGNÓSTICO 2 — formato bruto de id_escola nos dois lados")
print("=" * 70)

print(f"\nescola['id_escola'] (Bronze) — dtype: {escola['id_escola'].dtype}")
print(f"Amostra (10 primeiros valores): {amostra_escola}")
print("Tamanhos (nº de caracteres) mais comuns:")
print(escola["id_escola"].astype(str).str.len().value_counts().head().to_string())

print(f"\nalunos['id_escola'] (Silver) — dtype: {alunos['id_escola'].dtype}")
print(f"Amostra (10 primeiros valores): {amostra_alunos}")
print("Tamanhos (nº de caracteres) mais comuns:")
print(alunos["id_escola"].astype(str).str.len().value_counts().head().to_string())

### 4.2 A amostra de 10 deu uma pista: será mesmo diferença de formato, ou são dois sistemas de código diferentes?

Nos 10 valores que apareceram, os de `escola` (Bronze) começam com `24`, `33`, `35`, `41` — que batem com o código de UF de 2 dígitos do IBGE (24=RN, 33=RJ, 35=SP, 41=PR; o código de escola do INEP costuma seguir essa mesma convenção). Já os de `alunos` (Silver) começam **todos** com `60` — e `60` não é um código de UF válido no Brasil (o maior é 53, do DF). Isso é mais forte que "zero à esquerda perdido": sugere que os dois `id_escola` vêm de **sistemas de codificação diferentes**, não é só formatação.

Mas 10 linhas é pouco pra afirmar isso com confiança (pode ser só o efeito de as primeiras linhas do arquivo virem do mesmo município/partição, sem representar a base toda). Antes de concluir, olho a distribuição completa dos 2 primeiros dígitos nas duas tabelas inteiras.

In [ ]:
prefixo_2_digitos_escola = escola["id_escola"].str[:2].value_counts().sort_index()
prefixo_2_digitos_alunos = alunos["id_escola"].str[:2].value_counts().sort_index()

print(f"Prefixos de 2 dígitos em escola['id_escola'] (Bronze) — {prefixo_2_digitos_escola.shape[0]} valores distintos:")
print(prefixo_2_digitos_escola.to_string())

print(f"\nPrefixos de 2 dígitos em alunos['id_escola'] (Silver) — {prefixo_2_digitos_alunos.shape[0]} valores distintos:")
print(prefixo_2_digitos_alunos.to_string())

# códigos de UF válidos no IBGE vão de 11 a 53 - confiro se os prefixos de
# alunos caem fora desse intervalo de forma consistente (o que sugeriria um
# sistema de codificação diferente do usado em escola/município)
prefixos_fora_do_padrao_ibge = prefixo_2_digitos_alunos[
    ~prefixo_2_digitos_alunos.index.astype(int).isin(range(11, 54))
]
print(f"\nPrefixos de alunos['id_escola'] FORA da faixa de UF do IBGE (11-53): "
      f"{prefixos_fora_do_padrao_ibge.sum():,} linhas ({prefixos_fora_do_padrao_ibge.sum() / len(alunos) * 100:.1f}% da base)")

### 4.3 Conclusão (rodado em 2026-09-07): confirmado, são dois sistemas de código diferentes

**Resultado:** `escola` (Bronze) tem 27 prefixos de UF distintos (11, 12, 13... 43 — todos batendo com o padrão do IBGE, com volumes proporcionais ao tamanho de cada UF). Já **100% das 3.867.999 linhas de `alunos` têm `id_escola` começando em `60`**, sem uma única exceção, em todo o Brasil.

Isso não é mais suspeita, é confirmado: `id_escola` em `alunos` (`br_inep_avaliacao_alfabetizacao.alunos`) não usa o mesmo sistema de código de escola do INEP que a tabela `escola` (`br_inep_censo_escolar.escola`) usa. Os últimos dígitos variam de verdade (42.811 valores distintos, Seção 3), então o campo funciona como identificador — só que não é o registro oficial do Censo Escolar. Provavelmente é um código interno da própria fonte `br_inep_avaliacao_alfabetizacao` (um programa/avaliação específico), não o `co_entidade`/código INEP padrão.

**Isso não é um problema que eu resolvo só ajustando código** — é uma pergunta sobre a fonte de dados original (BigQuery/`basedosdados`): existe alguma outra coluna em `br_inep_avaliacao_alfabetizacao.alunos` com o código oficial do INEP (`co_entidade` ou parecido), que não foi trazida pro Silver na ingestão original? Fui investigar isso olhando o schema de origem e providenciando uma nova ingestão da tabela `escola` completa (455 colunas, ver Seção 7) — sem esse crosswalk, não tem como juntar `alunos` com `escola`/`escola_completo` por `id_escola`, e a Opção C fica sem viabilidade técnica até essa questão ser resolvida.

## 5. Correlação (Spearman) de cada coluna numérica da `escola` com `percentual_aprovados`

Como combinamos: uso `percentual_aprovados` (numérico, contínuo) como a métrica pra correlacionar, em vez de já converter em categoria "boa"/"ruim" — isso é mais informativo nesta fase de diagnóstico e não depende de eu já ter decidido o corte da pendência 11.

Só entram aqui as colunas **numéricas** (int/float/bool) da tabela `escola` — as colunas categóricas/texto (ex. `tipo_localizacao`, `rede`, `sigla_uf`) ficam de fora dessa correlação simples por enquanto; codificá-las (dummies) é parte do trabalho de feature engineering da pendência 13, não deste diagnóstico inicial. Considero só as escolas com alvo calculável (as que têm pelo menos 1 aluno avaliado).

In [ ]:
base_correlacao_escola = escola_com_alvo[escola_com_alvo["percentual_aprovados"].notna()].copy()

colunas_numericas_escola = base_correlacao_escola.select_dtypes(include=["number", "bool"]).columns.tolist()
colunas_numericas_escola = [c for c in colunas_numericas_escola if c != "percentual_aprovados"]

print(f"Colunas numéricas consideradas na correlação: {len(colunas_numericas_escola)} de {escola.shape[1]} totais")

correlacao_escola = (
    base_correlacao_escola[colunas_numericas_escola + ["percentual_aprovados"]]
    .corr(method="spearman")["percentual_aprovados"]
    .drop("percentual_aprovados")
    .sort_values(key=abs, ascending=False)
)

correlacao_escola_df = correlacao_escola.reset_index()
correlacao_escola_df.columns = ["coluna", "correlacao_spearman_com_percentual_aprovados"]

print(correlacao_escola_df.to_string(index=False))


### 5.1 Export em texto (CSV) da correlação, pra colar na conversa

In [ ]:
print(correlacao_escola_df.to_csv(index=False))


## 6. Próximos passos (depois de eu revisar os dois exports acima)

- Descartar (ou marcar como "tratar com cuidado") as colunas com % de nulo muito alto — o corte exato (50%? 80%?) eu decido olhando o que aparecer no export da Seção 2, não antes.
- Entre as colunas que sobrarem, ver quais fazem mais sentido como razão/proporção em vez de contagem bruta (pendência 13) — ex. `quantidade_matricula_feminino` sozinha diz pouco, a razão dela sobre o total de matrículas da escola diz mais.
- Cruzar a lista de correlação (Seção 5) com a de preenchimento (Seção 2) — uma coluna pode ter correlação alta só porque tem poucos dados preenchidos (ruído de amostra pequena), então as duas leituras precisam ser olhadas juntas, não uma sozinha.
- Só depois disso decidir a pendência 11 (definição final do alvo) com mais uma informação na mão: a distribuição real de `percentual_aprovados` por escola (ver `.describe()` da Seção 3), que ajuda a escolher o corte de "escola boa/ruim" com base em dado, não em chute.


## 7. Checagem do novo path `escola_completo/` (ingestão nova, ainda não vou carregar tudo)

Depois de identificar o problema de crosswalk da Seção 4, providenciei uma nova ingestão da tabela `escola` completa (455 colunas) num path novo: `bronze/br_inep_censo_escolar/escola_completo/`. Antes de decidir trabalhar com essa base, quero só confirmar que ela já está populada — **isso aqui é só um check** (lista os arquivos e o tamanho total no S3, não carrega nada pra memória). Só vou de fato ler e cachear essa base localmente depois de resolvermos a questão do `id_escola` (Seção 4.1) — carregar 455 colunas sem essa confirmação seria trabalho (e memória) gasto à toa se o mesmo problema de chave aparecer aqui também.

In [ ]:
PREFIXO_BRONZE_ESCOLA_COMPLETO = "bronze/br_inep_censo_escolar/escola_completo/"

paginador_completo = s3.get_paginator("list_objects_v2")
arquivos_escola_completo = []
for pagina in paginador_completo.paginate(Bucket=BUCKET, Prefix=PREFIXO_BRONZE_ESCOLA_COMPLETO):
    for objeto in pagina.get("Contents", []):
        if objeto["Key"].endswith(".parquet"):
            arquivos_escola_completo.append((objeto["Key"], objeto["Size"]))

if not arquivos_escola_completo:
    print(f"⚠️ Nada encontrado em s3://{BUCKET}/{PREFIXO_BRONZE_ESCOLA_COMPLETO} ainda.")
    print("A ingestão nova pode não ter terminado, ou o path/nome pode ser outro - preciso conferir antes de seguir.")
else:
    tamanho_total_mb = sum(tamanho for _, tamanho in arquivos_escola_completo) / 1e6
    print(f"✅ Path populado: {len(arquivos_escola_completo)} arquivo(s) .parquet encontrado(s)")
    print(f"Tamanho total: {tamanho_total_mb:,.1f} MB")
    print("\nArquivos:")
    for chave, tamanho in arquivos_escola_completo:
        print(f"  {chave} — {tamanho / 1e6:.2f} MB")

In [ ]:
# ⚠️ não rodar ainda - ver observação da célula de markdown acima
from pathlib import Path

CACHE_ESCOLA_COMPLETO = Path("../..") / "data" / "processed" / "escola_completo.parquet"

if CACHE_ESCOLA_COMPLETO.exists():
    print(f"Lendo do cache local: {CACHE_ESCOLA_COMPLETO}")
    escola_completo = pd.read_parquet(CACHE_ESCOLA_COMPLETO)
else:
    escola_completo = _ler_parquet_do_prefixo(BUCKET, PREFIXO_BRONZE_ESCOLA_COMPLETO)
    CACHE_ESCOLA_COMPLETO.parent.mkdir(parents=True, exist_ok=True)
    escola_completo.to_parquet(CACHE_ESCOLA_COMPLETO, index=False)
    print(f"Cache salvo em: {CACHE_ESCOLA_COMPLETO}")

print(f"escola_completo: {escola_completo.shape[0]:,} linhas, {escola_completo.shape[1]} colunas")
print(f"Uso de memória: {escola_completo.memory_usage(deep=True).sum() / 1e6:,.1f} MB")

### 7.1 Carga completa + cache local (não rodar ainda — só depois de decidir seguir com essa base)

Deixo essa célula pronta, mas ela **não faz parte do fluxo de agora** — 455 colunas é bastante dado, e só faz sentido carregar de verdade depois de resolvermos a questão do `id_escola` (Seção 4.1). Quando chegar a hora, ela lê do S3 e salva um cache local em Parquet, o mesmo padrão que o `load_data.py` já usa pra `base_modelagem.parquet` (`data/processed/`, que já está no `.gitignore`) — assim, depois da primeira vez, eu leio do disco local em vez de baixar do S3 de novo toda vez que reiniciar o kernel.